__Task1 - download data and preprocessing__

Import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from category_encoders import HashingEncoder

from collections import Counter
from typing import Optional, Tuple
from sklearn.metrics import roc_auc_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

Import data

In [2]:
df = pd.read_csv("data/training.csv")
print(df.shape)
df.head(2)

(72983, 34)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053


Filling gaps

In [3]:
df.isna().sum()

RefId                                    0
IsBadBuy                                 0
PurchDate                                0
Auction                                  0
VehYear                                  0
VehicleAge                               0
Make                                     0
Model                                    0
Trim                                  2360
SubModel                                 8
Color                                    8
Transmission                             9
WheelTypeID                           3169
WheelType                             3174
VehOdo                                   0
Nationality                              5
Size                                     5
TopThreeAmericanName                     5
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
MMRCurrentAuctionAveragePrice          315
MMRCurrentA

In [4]:
cat_features_with_missing = ['Trim', 'SubModel', 'Color', 'Size', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'AUCGUART', 'PRIMEUNIT']
for col in cat_features_with_missing:
    df[col] = df[col].fillna("missing")

In [5]:
numeric_features = ['WheelTypeID', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice']
for col in numeric_features:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

In [6]:
df.isna().sum()

RefId                                0
IsBadBuy                             0
PurchDate                            0
Auction                              0
VehYear                              0
VehicleAge                           0
Make                                 0
Model                                0
Trim                                 0
SubModel                             0
Color                                0
Transmission                         0
WheelTypeID                          0
WheelType                            0
VehOdo                               0
Nationality                          0
Size                                 0
TopThreeAmericanName                 0
MMRAcquisitionAuctionAveragePrice    0
MMRAcquisitionAuctionCleanPrice      0
MMRAcquisitionRetailAveragePrice     0
MMRAcquisitonRetailCleanPrice        0
MMRCurrentAuctionAveragePrice        0
MMRCurrentAuctionCleanPrice          0
MMRCurrentRetailAveragePrice         0
MMRCurrentRetailCleanPric

Splitting data by date in  ratio .33/.33/.33

In [7]:
date_threshold_1 = pd.to_datetime("2009-09-15")
date_threshold_2 = pd.to_datetime("2010-05-15")
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

df_train = df[df['PurchDate'] < date_threshold_1]
df_val = df[(df['PurchDate'] >= date_threshold_1) & (df['PurchDate'] < date_threshold_2)]
df_test = df[df['PurchDate'] >= date_threshold_2]

print(f"df_train: {df_train.shape}, percentile_shape : {int((df_train.shape[0]/df.shape[0])*100)}%")
print(f"df_val: {df_val.shape}, percentile_shape : {int(df_val.shape[0]/df.shape[0]*100)}%")
print(f"df_test: {df_test.shape}, percentile_shape : {int(df_test.shape[0]/df.shape[0]*100)}%")

df_train: (24232, 34), percentile_shape : 33%
df_val: (24486, 34), percentile_shape : 33%
df_test: (24265, 34), percentile_shape : 33%


In [8]:
print('Checking:')
if max(df_train['PurchDate']) < max(df_val['PurchDate']) < min(df_test['PurchDate']):
    print("train < val < test")
else: 
    print("Uvays")

df_train = df_train.drop(['PurchDate', 'RefId'], axis=1)
df_val = df_val.drop(['PurchDate', 'RefId'], axis=1)
df_test = df_test.drop(['PurchDate', 'RefId'], axis=1)

Checking:
train < val < test


Transformation of categorical features

In [9]:
df_cat_features = df_train.select_dtypes(include='object')
for cat in df_cat_features.columns:
    print(f"{cat}: {df[cat].nunique()}")

Auction: 3
Make: 33
Model: 1063
Trim: 135
SubModel: 864
Color: 17
Transmission: 4
WheelType: 4
Nationality: 5
Size: 13
TopThreeAmericanName: 5
PRIMEUNIT: 3
AUCGUART: 3
VNST: 37


OneHotEncoder : ['Auction', 'Transmission', 'WheelType', 'Nationality', ThreeAmericanName', 'PRIMEUNIT', 'AUCGUART'] \
Frequency encoding : ['Make', 'Color', 'Size', 'VNST'] \
Hashing encoding : ['Trim', 'Model', 'SubModel']

In [10]:
features_for_OHE = ['Auction', 'Transmission', 'WheelType', 'Nationality', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[features_for_OHE])

X_train = pd.DataFrame(ohe.transform(df_train[features_for_OHE]), columns=ohe.get_feature_names_out(features_for_OHE), index=df_train.index)
X_val   = pd.DataFrame(ohe.transform(df_val[features_for_OHE]),   columns=ohe.get_feature_names_out(features_for_OHE), index=df_val.index)
X_test  = pd.DataFrame(ohe.transform(df_test[features_for_OHE]),  columns=ohe.get_feature_names_out(features_for_OHE), index=df_test.index)

df_train = df_train.drop(columns=features_for_OHE).join(X_train)
df_val   = df_val.drop(columns=features_for_OHE).join(X_val)
df_test  = df_test.drop(columns=features_for_OHE).join(X_test)

In [11]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

(24232, 47)
(24486, 47)
(24265, 47)


In [12]:
cat_features = df_train.select_dtypes(include='object')
print(cat_features.columns)

Index(['Make', 'Model', 'Trim', 'SubModel', 'Color', 'Size', 'VNST'], dtype='object')


In [13]:
freq_features = ['Make', 'Color', 'Size', 'VNST']
for col in freq_features:
    freq = df_train[col].value_counts(normalize=True)
    df_train[col] = df_train[col].map(freq)
    df_val[col] = df_val[col].map(freq).fillna(0)
    df_test[col] = df_test[col].map(freq).fillna(0)

hash_cols = ['Trim', 'Model', 'SubModel']
hash_enc = HashingEncoder(cols=hash_cols, n_components=8)
hash_enc.fit(df_train[hash_cols])

df_train_hash = hash_enc.transform(df_train[hash_cols])
df_val_hash = hash_enc.transform(df_val[hash_cols])
df_test_hash = hash_enc.transform(df_test[hash_cols])

df_train = df_train.drop(columns=hash_cols).join(df_train_hash)
df_val = df_val.drop(columns=hash_cols).join(df_val_hash)
df_test = df_test.drop(columns=hash_cols).join(df_test_hash)

In [14]:
print(f"'train {df_train.shape}' - 'val {df_val.shape}' - 'test {df_test.shape}'")
df_train.sample(5)

'train (24232, 52)' - 'val (24486, 52)' - 'test (24265, 52)'


,IsBadBuy,VehYear,VehicleAge,Make,Color,WheelTypeID,VehOdo,Size,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,...,AUCGUART_GREEN,AUCGUART_missing,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7
31491,0,2008,1,0.196682,0.082948,2.0,49162,0.375743,8196.0,9044.0,...,0.0,1.0,0,0,3,0,0,0,0,0
37205,0,2004,5,0.022202,0.140806,1.0,85176,0.030043,7059.0,8319.0,...,0.0,1.0,1,0,0,0,1,0,1,0
61087,0,2003,6,0.232915,0.173159,2.0,78704,0.089592,2203.0,3337.0,...,0.0,1.0,2,0,0,1,0,0,0,0
35029,0,2005,4,0.196682,0.140806,1.0,77669,0.101519,6632.0,8224.0,...,0.0,1.0,0,0,0,0,0,1,1,1
69599,0,2005,4,0.196682,0.206091,2.0,87473,0.101519,3554.0,4694.0,...,0.0,1.0,0,0,1,0,0,1,1,0


In [15]:
X_train = df_train.drop('IsBadBuy', axis=1)
X_val = df_val.drop('IsBadBuy', axis=1)
X_test = df_test.drop('IsBadBuy', axis=1)

y_train = df_train['IsBadBuy']
y_val = df_val['IsBadBuy']
y_test = df_test['IsBadBuy']

__Task2 - Implementation of classes of Classification and Regression Trees__

In [16]:
class Node:
    def __init__(self, X, y, depth=0):
        self.X = X
        self.y = y
        self.depth = depth
        self.left: Optional[Node] = None
        self.right: Optional[Node] = None
        self.feature_index = None
        self.threshold = None
        self.prediction = self._compute_prediction()

    def _compute_prediction(self):
        raise NotImplementedError("Define in subclass.")

    def is_leaf(self):
        return self.left is None and self.right is None

class BaseTree:
    def __init__(self, max_depth=3, min_samples_split=2, randomized=False):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root: Optional[Node] = None
        self.randomized = randomized

    def fit(self, X, y):
        self.classes_ = sorted(set(y))
        self.root = self._grow_tree(X, y, depth=0)

    def predict(self, X):
        return np.array([self._predict_sample(self.root, x) for x in X])

    def _predict_sample(self, node: Node, x):
        while not node.is_leaf():
            if x[node.feature_index] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.prediction

    def _grow_tree(self, X, y, depth):
        node = self._NodeClass(X, y, depth)
        if (depth < self.max_depth) and (len(y) >= self.min_samples_split):
            feature_index, threshold = self._find_best_split(X, y)
            if feature_index is not None:
                idx_left = X[:, feature_index] <= threshold
                idx_right = ~idx_left
                if np.any(idx_left) and np.any(idx_right):
                    node.feature_index = feature_index
                    node.threshold = threshold
                    node.left = self._grow_tree(X[idx_left], y[idx_left], depth + 1)
                    node.right = self._grow_tree(X[idx_right], y[idx_right], depth + 1)
        return node

In [17]:
class ClassificationNode(Node):
    def _compute_prediction(self):
        counts = Counter(self.y)
        total = sum(counts.values())
        return {k: v / total for k, v in counts.items()}

class DecisionTreeClassifierCustom(BaseTree):
    def __init__(self, max_depth=3, min_samples_split=2, randomized=False):
        super().__init__(max_depth, min_samples_split, randomized)
        self._NodeClass = ClassificationNode

    def predict_proba(self, X):
        class_indices = {cls: i for i, cls in enumerate(self.classes_)}
        probas = np.zeros((len(X), len(self.classes_)))

        for i, x in enumerate(X):
            p = self._predict_sample(self.root, x)
            for cls, prob in p.items():
                probas[i, class_indices[cls]] = prob

        return probas

    def predict(self, X):
        probas = self.predict_proba(X)
        return np.array([max(p, key=p.get) for p in probas])

    def _gini(self, y):
        counts = Counter(y)
        total = len(y)
        return 1.0 - sum((v / total) ** 2 for v in counts.values())

    def _find_best_split(self, X, y) -> Tuple[Optional[int], Optional[float]]:
        best_index, best_threshold, best_impurity = None, None, float('inf')
        n_features = X.shape[1]
        features = np.random.choice(n_features, n_features if not self.randomized else max(1, int(np.sqrt(n_features))), replace=False)

        for feature_index in features:
            thresholds = np.unique(X[:, feature_index])
            if self.randomized:
                thresholds = np.random.choice(thresholds, min(len(thresholds), 5), replace=False)

            for threshold in thresholds:
                left = y[X[:, feature_index] <= threshold]
                right = y[X[:, feature_index] > threshold]
                if len(left) == 0 or len(right) == 0:
                    continue
                impurity = (len(left) * self._gini(left) + len(right) * self._gini(right)) / len(y)
                if impurity < best_impurity:
                    best_impurity = impurity
                    best_index = feature_index
                    best_threshold = threshold

        return best_index, best_threshold

In [18]:
class RegressionNode(Node):
    def _compute_prediction(self):
        return np.mean(self.y)

class DecisionTreeRegressorCustom(BaseTree):
    def __init__(self, max_depth=3, min_samples_split=2, randomized=False, max_features=None):
        super().__init__(max_depth, min_samples_split, randomized)
        self._NodeClass = RegressionNode
        self.max_features = max_features

    def _std(self, y):
        return np.var(y)

    def _find_best_split(self, X, y) -> Tuple[Optional[int], Optional[float]]:
        best_index, best_threshold, best_impurity = None, None, float('inf')
        n_features = X.shape[1]
        
        if self.randomized:
            max_feats = self.max_features or max(1, int(np.sqrt(n_features)))
            features = np.random.choice(n_features, max_feats, replace=False)
        else:
            features = np.arange(n_features)

        for feature_index in features:
            thresholds = np.unique(X[:, feature_index])
            if self.randomized:
                thresholds = np.random.choice(thresholds, min(len(thresholds), 5), replace=False)

            for threshold in thresholds:
                left = y[X[:, feature_index] <= threshold]
                right = y[X[:, feature_index] > threshold]
                if len(left) == 0 or len(right) == 0:
                    continue
                impurity = (len(left) * self._std(left) + len(right) * self._std(right)) / len(y)
                if impurity < best_impurity:
                    best_impurity = impurity
                    best_index = feature_index
                    best_threshold = threshold

        return best_index, best_threshold


__Task3 - Calculating Gini score by DecisionTreeClassifierCustom__

In [19]:
clf = DecisionTreeClassifierCustom(max_depth=5)
clf.fit(X_train.values, y_train)

probas_val_raw = clf.predict_proba(X_val.values)
probas_val = probas_val_raw[:, 1]

gini_score = 2 * roc_auc_score(y_val, probas_val) - 1
print(f"Gini score by DecisionTreeClassifierCustom on val data: {gini_score:.3f}")

Gini score by DecisionTreeClassifierCustom on val data: 0.421


__Task4 - Calculating Gini score by DecisionTreeClassifier from sklearn library__

In [20]:
clf_skl = DecisionTreeClassifier(max_depth=5)
clf_skl.fit(X_train.values, y_train)

probas_val_raw = clf_skl.predict_proba(X_val.values)
probas_val = np.array([p[1] for p in probas_val_raw])

gini_score = 2 * roc_auc_score(y_val, probas_val) - 1
print(f"Gini score by DecisionTreeClassifier on val data: {gini_score:.3f}")

Gini score by DecisionTreeClassifier on val data: 0.445


The Sklearn library tree gives better results because it uses optimized algorithms for finding the best partition, efficient implementation, and regularization, which improve the quality of the model.

__Task5 - Implementation class RandomForestClassifierCustom__

In [21]:
class RandomForestClassifierCustom:
    def __init__(self, n_estimators=100, max_depth=None, max_features="sqrt", random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.features_idx = []
        if random_state is not None:
            np.random.seed(random_state)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.trees = []
        self.features_idx = []
        
        for _ in range(self.n_estimators):
            idx = np.random.choice(n_samples, n_samples, replace=True)
            X_boot = X.iloc[idx].values
            y_boot = y.iloc[idx].values
            
            if self.max_features == "sqrt": n_feats = int(np.sqrt(n_features))
            elif self.max_features == "log2": n_feats = int(np.log2(n_features))
            else: n_feats = n_features
            
            feat_idx = np.random.choice(n_features, n_feats, replace=False)
            self.features_idx.append(feat_idx)
            
            tree = DecisionTreeClassifier(max_depth=self.max_depth, random_state=self.random_state)
            tree.fit(X_boot[:, feat_idx], y_boot)
            self.trees.append(tree)
    
    def predict_proba(self, X):
        preds = []
        for tree, feat_idx in zip(self.trees, self.features_idx):
            preds.append(tree.predict_proba(X[:, feat_idx]))
        return np.mean(preds, axis=0)
    
    def predict(self, X):
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)

In [22]:
rf = RandomForestClassifierCustom(n_estimators=50, max_depth=7, random_state=42)
rf.fit(X_train, y_train)

y_val_proba = rf.predict_proba(X_val.values)[:, 1]

gini_score = 2 * roc_auc_score(y_val, y_val_proba) - 1
print(f"Gini score by RandomForestCustom on val data: {gini_score:.3f}")

Gini score by RandomForestCustom on val data: 0.443


__Task6 - Implementation class GradientBoostingClassifierCustom__

In [23]:
class GradientBoostingClassifierCustom:
    def __init__(self, max_depth=3, number_of_trees=100, learning_rate=0.1, max_features=None, random_state=None):
        self.max_depth = max_depth
        self.number_of_trees = number_of_trees
        self.learning_rate = learning_rate
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.F0 = None
        if random_state:
            np.random.seed(random_state)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        X = np.asarray(X)
        n_samples = X.shape[0]
        p = np.mean(y)
        self.F0 = np.log(p / (1 - p))
        Fm = np.ones(n_samples) * self.F0

        for m in range(self.number_of_trees):
            prob = self._sigmoid(Fm)
            grad = y - prob
            tree = DecisionTreeRegressor()
            tree.fit(X, grad)
            self.trees.append(tree)
            Fm += self.learning_rate * tree.predict(X)

    def predict_proba(self, X):
        X = np.asarray(X)
        Fm = np.ones(X.shape[0]) * self.F0
        for tree in self.trees:
            Fm += self.learning_rate * tree.predict(X)
        prob = self._sigmoid(Fm)
        return np.vstack([1 - prob, prob]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

In [24]:
gb = GradientBoostingClassifierCustom(max_depth=3, number_of_trees=50, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

y_val_proba = gb.predict_proba(X_val)[:, 1]

gini_score = 2 * roc_auc_score(y_val, y_val_proba) - 1
print(f"Gini score by GradientBoostingClassifierCustom on val data: {gini_score:.3f}")

Gini score by GradientBoostingClassifierCustom on val data: 0.233


__Task7 - Comparing LightGBM, CatBoost, XGBoost algorithms__

In [25]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=7,
    random_state=42,
    verbosity=-1
)
lgb_model.fit(X_train, y_train, )
y_pred_lgb = lgb_model.predict_proba(X_val)[:, 1]

In [26]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict_proba(X_val)[:, 1]

In [27]:
cat_model = cb.CatBoostClassifier(
    iterations=100,
    learning_rate=0.1,
    depth=4,
    verbose=0,
    random_state=42
)
cat_model.fit(X_train, y_train)
y_pred_catb = cat_model.predict_proba(X_val)[:, 1]

In [28]:
gini_score_lgb = 2 * roc_auc_score(y_val, y_pred_lgb) - 1
gini_score_xgb = 2 * roc_auc_score(y_val, y_pred_xgb) - 1
gini_score_catb = 2 * roc_auc_score(y_val, y_pred_catb) - 1

print(f"Gini score by LGBM on val data: {gini_score_lgb:.3f}")
print(f"Gini score by XGB on val data: {gini_score_xgb:.3f}")
print(f"Gini score by CatBoost on val data: {gini_score_catb:.3f}")

Gini score by LGBM on val data: 0.476
Gini score by XGB on val data: 0.470
Gini score by CatBoost on val data: 0.493


The best gini score with CatBoost model. Under the hood, Catboost uses category feature encoding to prevent overfitting. Dart in XGBoost is a dropout method like neural networks for excluding overfitting of model.

__Task8 - Checking Gini score for the best model__

In [29]:
y_train_pred = cat_model.predict_proba(X_train)[:, 1]
y_val_pred = cat_model.predict_proba(X_val)[:, 1]
y_test_pred = cat_model.predict_proba(X_test)[:, 1]

gini_train = 2 * roc_auc_score(y_train, y_train_pred) - 1
gini_val = 2 * roc_auc_score(y_val, y_val_pred) - 1
gini_test = 2 * roc_auc_score(y_test, y_test_pred) - 1

print(f"Gini (Train): {gini_train:.3f}")
print(f"Gini (Val):   {gini_val:.3f}")
print(f"Gini (Test):  {gini_test:.3f}")

Gini (Train): 0.554
Gini (Val):   0.493
Gini (Test):  0.491


By reducing the depth of the trees, we were able to reduce overfitting and improve the generalization ability of the model. The difference between the training and test metrics is small, so we can say that the model is not overfitted and shows stable performance on new data.

__Task9 - Implementation of ExtraTreesClassifierCustom__

In [ ]:
class ExtraTreesClassifierCustom:
    def __init__(self, n_estimators=100, max_depth=None, max_features="sqrt", random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []
        self.features_idx = []
        if random_state:
            np.random.seed(random_state)

    def fit(self, X, y):
        X = np.asarray(X)
        y = np.asarray(y)
        self.classes_ = sorted(set(y))
        _, n_features = X.shape

        for _ in range(self.n_estimators):
            if self.max_features == "sqrt": n_feats = int(np.sqrt(n_features))
            elif self.max_features == "log2": n_feats = int(np.log2(n_features))
            else: n_feats = n_features

            feat_idx = np.random.choice(n_features, n_feats, replace=False)
            self.features_idx.append(feat_idx)
            
            tree = DecisionTreeClassifierCustom(
                max_depth=self.max_depth,
                randomized=True
            )

            tree.fit(X[:, feat_idx], y)
            self.trees.append(tree)

    def predict_proba(self, X):
        X = np.asarray(X)
        all_probas = []

        for tree, feat_idx in zip(self.trees, self.features_idx):
            probas = tree.predict_proba(X[:, feat_idx])
            all_probas.append(probas)

        return np.mean(all_probas, axis=0)

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

In [34]:
clf = ExtraTreesClassifierCustom(n_estimators=50, max_depth=7, random_state=42)
clf.fit(X_train.values, y_train.values)
y_pred = clf.predict_proba(X_val)[:, 1]
gini = 2 * roc_auc_score(y_val, y_pred) - 1
print(f"Gini score by ExtraTreesCustom on val data: {gini:.3f}")

Gini score by ExtraTreesCustom on val data: 0.455
